In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
!pip install -q faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.7 MB/s eta 0:00:00:00:0100:01


In [10]:
import re
import string

from glob import glob
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import torch
import faiss

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
import os
import wandb

wandb_mode = "online" if os.environ.get("WANDB_API_KEY") else "disabled"

run = wandb.init(
    project="23f2000775-t22026",
    name="milestone3-rag",
    mode=wandb_mode,
    config={
        "embedding_model": embedding_model_name,
        "llm": llm_name,
        "top_k": top_k,
        "sample_size": sample_size
    },
    reinit=True
)

wandb.log({
    "MAP@3": rag_map3,
    "Accuracy": rag_accuracy
})

run.finish()

In [5]:
train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

train = pd.read_csv(train_path)

print(train.shape)
train.head()

(2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [ ]:
train.isnull().sum()

In [ ]:
text_columns = ['prompt', 'A', 'B', 'C', 'D', 'E']

train[text_columns] = train[text_columns].fillna('')

In [ ]:
def clean_text(text):
    text = text.lower()

    for mark in string.punctuation:
        text = text.replace(mark, ' ')

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
for column in text_columns:
    train[column + '_clean'] = train[column].apply(clean_text)

train[['prompt', 'prompt_clean']].head()

In [ ]:
train['prompt_tokens'] = train['prompt_clean'].str.split()

train[['prompt_clean', 'prompt_tokens']].head()

In [ ]:
def average_precision_at_3(actual, predicted):
    predicted = predicted[:3]

    if actual not in predicted:
        return 0

    rank = predicted.index(actual) + 1

    return 1 / rank

In [ ]:
def map_at_3(actual_answers, predictions):
    scores = []

    for actual, predicted in zip(actual_answers, predictions):
        score = average_precision_at_3(actual, predicted)
        scores.append(score)

    return np.mean(scores)

In [ ]:
clean_columns = [
    'prompt_clean',
    'A_clean',
    'B_clean',
    'C_clean',
    'D_clean',
    'E_clean'
]

all_text = []

for column in clean_columns:
    all_text.extend(train[column].tolist())

print('Number of text samples:', len(all_text))

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf.fit(all_text)

print('TF-IDF vocabulary size:', len(tfidf.vocabulary_))

In [ ]:
option_labels = ['A', 'B', 'C', 'D', 'E']
tfidf_predictions = []
tfidf_scores = []

for _, row in train.iterrows():
    prompt_vector = tfidf.transform([row['prompt_clean']])

    options = [
        row['A_clean'],
        row['B_clean'],
        row['C_clean'],
        row['D_clean'],
        row['E_clean']
    ]

    option_vectors = tfidf.transform(options)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    tfidf_predictions.append(prediction)
    tfidf_scores.append(similarities)

In [ ]:
train['tfidf_prediction'] = tfidf_predictions

tfidf_map = map_at_3(
    train['answer'].tolist(),
    tfidf_predictions
)

print('TF-IDF MAP@3:', round(tfidf_map, 4))

train[
    ['id', 'answer', 'tfidf_prediction']
].head(10)

In [ ]:
sentences = []

for column in clean_columns:
    for text in train[column]:
        sentences.append(text.split())

print('Number of sentences:', len(sentences))

In [ ]:
word2vec = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

print('Word2Vec vocabulary size:', len(word2vec.wv))

In [ ]:
def get_sentence_vector(text):
    vectors = []

    for word in text.split():
        if word in word2vec.wv:
            vectors.append(word2vec.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [ ]:
word2vec_predictions = []
word2vec_scores = []

for _, row in train.iterrows():
    prompt_vector = get_sentence_vector(
        row['prompt_clean']
    ).reshape(1, -1)

    option_vectors = []

    for label in option_labels:
        option_vector = get_sentence_vector(
            row[label + '_clean']
        )
        option_vectors.append(option_vector)

    option_vectors = np.array(option_vectors)

    similarities = cosine_similarity(
        prompt_vector,
        option_vectors
    )[0]

    order = np.argsort(similarities)[::-1]
    prediction = [option_labels[i] for i in order]

    word2vec_predictions.append(prediction)
    word2vec_scores.append(similarities)

In [ ]:
train['word2vec_prediction'] = word2vec_predictions

word2vec_map = map_at_3(
    train['answer'].tolist(),
    word2vec_predictions
)

print('Word2Vec MAP@3:', round(word2vec_map, 4))

## milestone 3

knowledge dataset loaded

In [ ]:
knowledge_data = load_dataset("allenai/openbookqa", "additional", split="train")

documents = []
for row in knowledge_data:
    fact = row.get("fact1", "")
    if fact and fact not in documents:
        documents.append(fact)

documents = documents[:2000]

print("Number of knowledge documents:", len(documents))
print(documents[0])

In [ ]:
embedding_model_name = "BAAI/bge-small-en-v1.5"
embedding_model = SentenceTransformer(embedding_model_name)

document_embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(document_embeddings.shape)

In [ ]:
embedding_dim = document_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(document_embeddings.astype("float32"))

print("FAISS index size:", faiss_index.ntotal)


In [ ]:
#retrieval:
top_k = 3

prompt_embeddings = embedding_model.encode(
    train["prompt"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

scores, indices = faiss_index.search(prompt_embeddings, top_k)

retrieved_documents = []
for row_indices in indices:
    docs = [documents[i] for i in row_indices]
    retrieved_documents.append(docs)

train["retrieved_documents"] = retrieved_documents
train["retrieved_context"] = ["\n".join(docs) for docs in retrieved_documents]

print("Question:")
print(train.loc[0, "prompt"])
print("\nRetrieved context:")
print(train.loc[0, "retrieved_context"])


In [ ]:
def build_augmented_prompt(row):
    return f"""Context:
{row['retrieved_context']}

Question:
{row['prompt']}

Options:
A {row['A']}
B {row['B']}
C {row['C']}
D {row['D']}
E {row['E']}

Return only the best answer."""

In [ ]:
llm_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm = llm.to(device)

print("Using device:", device)

In [ ]:
sample_size = 25
sample = train.head(sample_size).copy()

generated_answers = []
rag_predictions = []

def get_answer_order(generated_text, row):
    text = generated_text.strip().upper()

    for label in option_labels:
        if text == label or text.startswith(label + " "):
            best = label
            break
    else:
        best = None
        for label in option_labels:
            option_text = str(row[label]).strip().upper()
            if option_text and (option_text in text or text in option_text):
                best = label
                break

    if best is None:
        best = "A"

    return [best] + [label for label in option_labels if label != best]

for _, row in sample.iterrows():
    prompt = build_augmented_prompt(row)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    outputs = llm.generate(
        **inputs,
        max_new_tokens=12
    )

    answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = get_answer_order(answer_text, row)

    generated_answers.append(answer_text)
    rag_predictions.append(prediction)

print("Finished inference for", sample_size, "questions")

In [ ]:
actual_answers = sample["answer"].tolist()

rag_map3 = map_at_3(actual_answers, rag_predictions)
rag_accuracy = np.mean([
    actual == predicted[0]
    for actual, predicted in zip(actual_answers, rag_predictions)
])

print("MAP@3:", rag_map3)
print("Accuracy:", rag_accuracy)

for i in range(min(3, len(sample))):
    print("\nExample", i + 1)
    print("Retrieved context:")
    print(sample.iloc[i]["retrieved_context"])
    print("Generated answer:", generated_answers[i])
    print("Predicted answer:", rag_predictions[i][0])
    print("Actual answer:", actual_answers[i])

In [ ]:
import os

input_files = []
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        input_files.append(os.path.join(dirname, filename))

test_path = [p for p in input_files if p.endswith("test.csv")][0]
sample_path = [p for p in input_files if p.endswith("sample_submission.csv")][0]

test = pd.read_csv(test_path)
submission = pd.read_csv(sample_path)

test_embeddings = embedding_model.encode(
    test["prompt"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

_, indices = faiss_index.search(test_embeddings, top_k)

test["retrieved_documents"] = [[documents[i] for i in row] for row in indices]
test["retrieved_context"] = ["\n".join(docs) for docs in test["retrieved_documents"]]

test_predictions = []

for _, row in test.iterrows():
    prompt = build_augmented_prompt(row)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = llm.generate(**inputs, max_new_tokens=12)
    answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prediction = get_answer_order(answer_text, row)
    test_predictions.append(" ".join(prediction[:3]))

submission.iloc[:, -1] = test_predictions
submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("Saved /kaggle/working/submission.csv")

In [ ]:
#milestonme 4 google forms

In [6]:


label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_map)

print("Encoded label at index 150:", train.loc[150, "label"])

Encoded label at index 150: 2


In [7]:
formatted_input = (
    str(train.loc[0, "prompt"]) +
    " [SEP] " +
    str(train.loc[0, "B"])
)

print(formatted_input)
print("Character length:", len(formatted_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character length: 407


In [11]:

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)

print(input_ids.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

torch.Size([1, 5, 128])


In [12]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

all_choices = []

for i in range(16):
    row = train.loc[i]

    choices = [
        str(row["prompt"]) + " [SEP] " + str(row["A"]),
        str(row["prompt"]) + " [SEP] " + str(row["B"]),
        str(row["prompt"]) + " [SEP] " + str(row["C"]),
        str(row["prompt"]) + " [SEP] " + str(row["D"]),
        str(row["prompt"]) + " [SEP] " + str(row["E"]),
    ]

    all_choices.append(choices)

flat_choices = [choice for row in all_choices for choice in row]

encoding = tokenizer(
    flat_choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].view(16, 5, 128)

print(input_ids.shape)

total_positions = input_ids.numel()
print("Total token positions:", total_positions)

torch.Size([16, 5, 128])
Total token positions: 10240


In [14]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"])
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

print(outputs.logits)
print(outputs.logits.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[0.1814, 0.1646, 0.1543, 0.1866, 0.1567]], grad_fn=<ViewBackward0>)
torch.Size([1, 5])


In [15]:
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"])
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

label = torch.tensor([label_map[row["answer"]]])

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=label
)

print(outputs.loss)
print(outputs.loss.shape)
print(outputs.loss.dim())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor(1.6120, grad_fn=<NllLossBackward0>)
torch.Size([])
0


In [16]:
!pip install -q peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.9 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

In [17]:
import torch
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", trainable_params)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 295681


In [19]:

from datasets import Dataset
label_map = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

rows = []

for _, row in train.head(100).iterrows():

    choices = [
        str(row["prompt"]) + " [SEP] " + str(row["A"]),
        str(row["prompt"]) + " [SEP] " + str(row["B"]),
        str(row["prompt"]) + " [SEP] " + str(row["C"]),
        str(row["prompt"]) + " [SEP] " + str(row["D"]),
        str(row["prompt"]) + " [SEP] " + str(row["E"]),
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    rows.append({
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": label_map[row["answer"]]
    })

dataset = Dataset.from_list(rows)

print(dataset)
print(len(dataset[0]["input_ids"]))

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})
5


In [20]:
model.eval()

row = train.loc[0]

choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}"
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

with torch.no_grad():
    outputs = model(
        input_ids=encoding["input_ids"].unsqueeze(0),
        attention_mask=encoding["attention_mask"].unsqueeze(0)
    )

probs = torch.softmax(outputs.logits, dim=1)

print(probs)
print("Probability of Option E:", round(probs[0, 4].item(), 4))

tensor([[0.2076, 0.1905, 0.1946, 0.2013, 0.2061]])
Probability of Option E: 0.2061
